# Import Primary Libraries

In [304]:
import pandas as pd
import numpy as np
import joblib
import sys
from pathlib import Path

# Import functions from src/features.py
sys.path.append(
    str(Path().resolve().parent)
)
from src.features import (
    team_rating,
    assign_tournament_weight
)

# Load Data and Model

In [305]:
fixtures = pd.read_csv("../data/processed/wc2026_matches.csv")
elo_ratings = pd.read_csv("../data/processed/elo_clean.csv")
players = pd.read_csv("../data/processed/players_clean.csv")
matches = pd.read_csv("../data/processed/matches.csv")

model = joblib.load("best_model.pkl")

In [306]:
# Renaming country column to host_country for clarity
fixtures.rename(columns={"country": "host_country"}, inplace=True)

# Build Features

## Squad Ratings
The squad ratings are derived from FC26. To prevent data leakage, they were not used as features in model training. Thus, they will not directly be used as features during simulation as well. 

In [307]:
squad_requirements = { # General squad requirements
    "GK": 3,
    "DEF": 9,
    "MID": 7,
    "FWD": 7
}
squad_ratings = team_rating(players, requirements=squad_requirements, rating_name="squad_rating")

top11_requirements = { # 4-3-3 formation as estimate for starting 11 requirements
    "GK": 1,
    "DEF": 4,
    "MID": 3,
    "FWD": 3
}
top11_ratings = team_rating(players, requirements=top11_requirements, rating_name="top11_rating")

# Rename columns before merger
squad_ratings.rename(columns={"avg_rating": "squad_rating"}, inplace=True)
top11_ratings.rename(columns={"avg_rating": "top11_rating"}, inplace=True)

# Merge ratings with fixtures
fixtures = fixtures.merge(
    squad_ratings[["country", "squad_rating"]], 
    left_on="home_team", 
    right_on="country", 
    how="left").rename(columns={"squad_rating": "home_squad_rating"}).drop(columns=["country"])

fixtures = fixtures.merge(
    squad_ratings[["country", "squad_rating"]], 
    left_on="away_team", 
    right_on="country", 
    how="left").rename(columns={"squad_rating": "away_squad_rating"}).drop(columns=["country"])

fixtures = fixtures.merge(
    top11_ratings[["country", "top11_rating"]], 
    left_on="home_team", 
    right_on="country", 
    how="left").rename(columns={"top11_rating": "home_top11_rating"}).drop(columns=["country"])

fixtures = fixtures.merge(
    top11_ratings[["country", "top11_rating"]], 
    left_on="away_team", 
    right_on="country", 
    how="left").rename(columns={"top11_rating": "away_top11_rating"}).drop(columns=["country"])

fixtures

,date,home_team,away_team,home_score,away_score,tournament,city,host_country,neutral,home_squad_rating,away_squad_rating,home_top11_rating,away_top11_rating
0,2026-06-11,Mexico,South Africa,NaN,NaN,FIFA World Cup,Mexico City,Mexico,False,75.576923,59.384615,77.181818,66.818182
1,2026-06-11,South Korea,Czech Republic,NaN,NaN,FIFA World Cup,Zapopan,Mexico,True,73.576923,75.576923,76.090909,77.090909
2,2026-06-12,Canada,Bosnia and Herzegovina,NaN,NaN,FIFA World Cup,Toronto,Canada,False,72.884615,72.807692,77.000000,76.272727
3,2026-06-12,United States,Paraguay,NaN,NaN,FIFA World Cup,Inglewood,United States,False,76.423077,73.692308,79.090909,75.181818
4,2026-06-13,Qatar,Switzerland,NaN,NaN,FIFA World Cup,Santa Clara,United States,True,68.307692,77.615385,72.090909,80.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,2026-06-27,Jordan,Argentina,NaN,NaN,FIFA World Cup,Arlington,United States,True,52.346154,82.115385,55.545455,84.454545
68,2026-06-27,Colombia,Portugal,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True,77.346154,81.961538,79.454545,84.636364
69,2026-06-27,DR Congo,Uzbekistan,NaN,NaN,FIFA World Cup,Atlanta,United States,True,71.384615,53.730769,75.727273,58.818182
70,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,East Rutherford,United States,True,59.500000,83.461538,68.363636,85.727273


## Home Team Host Status

In [308]:
fixtures["home_is_host"] = (fixtures["home_team"] == fixtures["host_country"]).astype(int)

## Elo features

In [309]:
# Checking for any discrepancies in country names between fixtures and elo ratings
fixtures_countries = set(fixtures["home_team"]).union(
    set(fixtures["away_team"])
)

elo_countries = set(elo_ratings["country_full"])

print("In fixtures but not elo:")
print(sorted(fixtures_countries - elo_countries))

print("\nIn elo but not fixtures:")
print(sorted(elo_countries - fixtures_countries))

In fixtures but not elo:
[]

In elo but not fixtures:
['Afghanistan', 'Albania', 'American Samoa', 'Andorra', 'Angola', 'Anguilla', 'Antigua and Barbuda', 'Armenia', 'Aruba', 'Azerbaijan', 'Bahamas', 'Bahrain', 'Bangladesh', 'Barbados', 'Belarus', 'Belize', 'Benin', 'Bermuda', 'Bhutan', 'Bolivia', 'Botswana', 'British Virgin Islands', 'Brunei Darussalam', 'Bulgaria', 'Burkina Faso', 'Burundi', 'Cambodia', 'Cameroon', 'Cayman Islands', 'Central African Republic', 'Chad', 'Chile', 'China PR', 'Chinese Taipei', 'Comoros', 'Congo', 'Cook Islands', 'Costa Rica', 'Cuba', 'Cyprus', 'Czechoslovakia', 'Denmark', 'Djibouti', 'Dominica', 'Dominican Republic', 'El Salvador', 'Equatorial Guinea', 'Eritrea', 'Estonia', 'Eswatini', 'Ethiopia', 'Faroe Islands', 'Fiji', 'Finland', 'Gabon', 'Georgia', 'Gibraltar', 'Greece', 'Grenada', 'Guam', 'Guatemala', 'Guinea', 'Guinea-Bissau', 'Guyana', 'Honduras', 'Hong Kong, China', 'Hungary', 'Iceland', 'India', 'Indonesia', 'Israel', 'Italy', 'Jamaica', 'Kazakh

In [310]:
# Filtering for the latest elo ratings for countries in the World Cup 2026
elo_ratings_wc2026 = elo_ratings[elo_ratings["country_full"].isin(fixtures_countries)]
elo_ratings_wc2026 = elo_ratings_wc2026[elo_ratings_wc2026["rank_date"] == "2026-04-01"] # last update to elo ratings before the World Cup 2026
print(len(elo_ratings_wc2026)) # ensuring that all countries are included

48


In [311]:
# Adding home and away elo ratings to fixtures dataframe
fixtures = fixtures.merge(
    elo_ratings_wc2026[["country_full", "total_points"]],
    left_on="home_team",
    right_on="country_full",
    how="left"
).rename(columns={"total_points": "home_elo"}).drop(columns=["country_full"])

fixtures = fixtures.merge(
    elo_ratings_wc2026[["country_full", "total_points"]],
    left_on="away_team",
    right_on="country_full",
    how="left"
).rename(columns={"total_points": "away_elo"}).drop(columns=["country_full"])

# Calculating elo difference and absolute elo difference
fixtures["elo_diff"] = fixtures["home_elo"] - fixtures["away_elo"] 
fixtures["abs_elo_diff"] = fixtures["elo_diff"].abs()

# Remove home_elo and away_elo as they are not needed for the model
fixtures.drop(columns=["home_elo", "away_elo"], inplace=True) 

fixtures

,date,home_team,away_team,home_score,away_score,tournament,city,host_country,neutral,home_squad_rating,away_squad_rating,home_top11_rating,away_top11_rating,home_is_host,elo_diff,abs_elo_diff
0,2026-06-11,Mexico,South Africa,NaN,NaN,FIFA World Cup,Mexico City,Mexico,False,75.576923,59.384615,77.181818,66.818182,1,334.0,334.0
1,2026-06-11,South Korea,Czech Republic,NaN,NaN,FIFA World Cup,Zapopan,Mexico,True,73.576923,75.576923,76.090909,77.090909,0,26.0,26.0
2,2026-06-12,Canada,Bosnia and Herzegovina,NaN,NaN,FIFA World Cup,Toronto,Canada,False,72.884615,72.807692,77.000000,76.272727,1,190.0,190.0
3,2026-06-12,United States,Paraguay,NaN,NaN,FIFA World Cup,Inglewood,United States,False,76.423077,73.692308,79.090909,75.181818,1,-112.0,112.0
4,2026-06-13,Qatar,Switzerland,NaN,NaN,FIFA World Cup,Santa Clara,United States,True,68.307692,77.615385,72.090909,80.000000,0,-464.0,464.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,2026-06-27,Jordan,Argentina,NaN,NaN,FIFA World Cup,Arlington,United States,True,52.346154,82.115385,55.545455,84.454545,0,-423.0,423.0
68,2026-06-27,Colombia,Portugal,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True,77.346154,81.961538,79.454545,84.636364,0,-9.0,9.0
69,2026-06-27,DR Congo,Uzbekistan,NaN,NaN,FIFA World Cup,Atlanta,United States,True,71.384615,53.730769,75.727273,58.818182,0,-72.0,72.0
70,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,East Rutherford,United States,True,59.500000,83.461538,68.363636,85.727273,0,-283.0,283.0


## Recent Form Features

In [312]:
# Creating a unified dataframe of the latest team features for both home and away teams
latest_team_features = pd.concat([
    matches[[
        "date",
        "home_team",
        "home_recent_win_rate",
        "home_recent_draw_rate",
        "home_avg_goals_last10",
        "home_avg_conceded_last10"
    ]].rename(columns={
        "home_team": "team",
        "home_recent_win_rate": "recent_win_rate",
        "home_recent_draw_rate": "recent_draw_rate",
        "home_avg_goals_last10": "avg_goals_last10",
        "home_avg_conceded_last10": "avg_conceded_last10"
    }),
    matches[[
        "date",
        "away_team",
        "away_recent_win_rate",
        "away_recent_draw_rate",
        "away_avg_goals_last10",
        "away_avg_conceded_last10"
    ]].rename(columns={
        "away_team": "team",
        "away_recent_win_rate": "recent_win_rate",
        "away_recent_draw_rate": "recent_draw_rate",
        "away_avg_goals_last10": "avg_goals_last10",
        "away_avg_conceded_last10": "avg_conceded_last10"
    })
])

# Extracting the latest features for each country
latest_team_features = (
    latest_team_features
    .sort_values("date")
    .groupby("team")
    .tail(1)
)

In [313]:
# Merging the latest team features with the fixtures dataframe for home teams
fixtures = fixtures.merge(
    latest_team_features,
    left_on="home_team",
    right_on="team",
    how="left"
)

# Renaming columns to be specific to home team and dropping redundant columns
fixtures = fixtures.rename(columns={
    "recent_win_rate": "home_recent_win_rate",
    "recent_draw_rate": "home_recent_draw_rate",
    "avg_goals_last10": "home_avg_goals_last10",
    "avg_conceded_last10": "home_avg_conceded_last10"
})
fixtures.drop(columns=["team", "date_y"], inplace=True)

# Merging the latest team features with the fixtures dataframe for away teams
fixtures = fixtures.merge(
    latest_team_features,
    left_on="away_team",
    right_on="team",
    how="left"
)

# Renaming columns to be specific to away team and dropping redundant columns
fixtures = fixtures.rename(columns={
    "recent_win_rate": "away_recent_win_rate",
    "recent_draw_rate": "away_recent_draw_rate",
    "avg_goals_last10": "away_avg_goals_last10",
    "avg_conceded_last10": "away_avg_conceded_last10"
})
fixtures.drop(columns=["team", "date"], inplace=True)

# Rename date_x back to date
fixtures.rename(columns={"date_x": "date"}, inplace=True)
fixtures

,date,home_team,away_team,home_score,away_score,tournament,city,host_country,neutral,home_squad_rating,...,elo_diff,abs_elo_diff,home_recent_win_rate,home_recent_draw_rate,home_avg_goals_last10,home_avg_conceded_last10,away_recent_win_rate,away_recent_draw_rate,away_avg_goals_last10,away_avg_conceded_last10
0,2026-06-11,Mexico,South Africa,NaN,NaN,FIFA World Cup,Mexico City,Mexico,False,75.576923,...,334.0,334.0,0.4,0.4,1.1,0.8,0.4,0.3,1.5,1.1
1,2026-06-11,South Korea,Czech Republic,NaN,NaN,FIFA World Cup,Zapopan,Mexico,True,73.576923,...,26.0,26.0,0.6,0.1,1.8,1.2,0.4,0.4,1.8,1.2
2,2026-06-12,Canada,Bosnia and Herzegovina,NaN,NaN,FIFA World Cup,Toronto,Canada,False,72.884615,...,190.0,190.0,0.4,0.5,1.1,0.4,0.4,0.4,2.1,1.1
3,2026-06-12,United States,Paraguay,NaN,NaN,FIFA World Cup,Inglewood,United States,False,76.423077,...,-112.0,112.0,0.5,0.1,1.7,1.6,0.4,0.3,1.1,1.0
4,2026-06-13,Qatar,Switzerland,NaN,NaN,FIFA World Cup,Santa Clara,United States,True,68.307692,...,-464.0,464.0,0.1,0.4,0.9,1.9,0.6,0.3,2.5,0.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,2026-06-27,Jordan,Argentina,NaN,NaN,FIFA World Cup,Arlington,United States,True,52.346154,...,-423.0,423.0,0.5,0.2,2.0,1.5,0.8,0.1,2.1,0.4
68,2026-06-27,Colombia,Portugal,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True,77.346154,...,-9.0,9.0,0.5,0.3,2.1,1.0,0.6,0.3,2.9,1.2
69,2026-06-27,DR Congo,Uzbekistan,NaN,NaN,FIFA World Cup,Atlanta,United States,True,71.384615,...,-72.0,72.0,0.7,0.2,1.2,0.3,0.4,0.5,1.5,0.7
70,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,East Rutherford,United States,True,59.500000,...,-283.0,283.0,0.4,0.4,1.5,1.4,0.8,0.1,2.5,0.4


In [314]:
fixtures["recent_form_diff"] = fixtures["home_recent_win_rate"] - fixtures["away_recent_win_rate"]
fixtures["abs_recent_form_diff"] = fixtures["recent_form_diff"].abs()
fixtures["recent_draw_diff"] = fixtures["home_recent_draw_rate"] - fixtures["away_recent_draw_rate"]
fixtures["abs_recent_draw_diff"] = fixtures["recent_draw_diff"].abs()
fixtures["diff_in_avg_goals"] = fixtures["home_avg_goals_last10"] - fixtures["away_avg_goals_last10"]
fixtures["diff_in_avg_conceded"] = fixtures["home_avg_conceded_last10"] - fixtures["away_avg_conceded_last10"]

## Tournament Weight

In [315]:
fixtures["tournament_weight"] = fixtures["tournament"].apply(assign_tournament_weight)

In [316]:
fixtures.info()

<class 'pandas.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 31 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   date                      72 non-null     str    
 1   home_team                 72 non-null     str    
 2   away_team                 72 non-null     str    
 3   home_score                0 non-null      float64
 4   away_score                0 non-null      float64
 5   tournament                72 non-null     str    
 6   city                      72 non-null     str    
 7   host_country              72 non-null     str    
 8   neutral                   72 non-null     bool   
 9   home_squad_rating         72 non-null     float64
 10  away_squad_rating         72 non-null     float64
 11  home_top11_rating         72 non-null     float64
 12  away_top11_rating         72 non-null     float64
 13  home_is_host              72 non-null     int64  
 14  elo_diff               

# Generate Match Probabilities

In [317]:
# Selecting the features for the model
feature_cols = [
    "neutral",
    "home_is_host",
    "elo_diff",
    "abs_elo_diff",
    "recent_form_diff",
    "abs_recent_form_diff",
    "home_recent_draw_rate",
    "away_recent_draw_rate",
    "recent_draw_diff",
    "abs_recent_draw_diff",
    "home_avg_goals_last10",
    "away_avg_goals_last10",
    "diff_in_avg_goals",
    "home_avg_conceded_last10",
    "away_avg_conceded_last10",
    "diff_in_avg_conceded",
    "tournament_weight"
]

X_wc = fixtures[feature_cols]

In [318]:
probs = model.predict_proba(X_wc)